# L4: Declarative Generative UI

In this lesson, you'll define a catalog of reusable UI building blocks and let the agent compose them into rich interfaces — dashboards, flight carousels, and more — without hand-coding each layout.

## 📋 Learning Objectives

1. **Learn the A2UI spec** - A declarative generative UI specification built by Google in collaboration with CopilotKit
2. **Define component catalogs** - Create definitions and renderers that the agent can compose at runtime
3. **Use dynamic and fixed schemas** - Compare agent-generated layouts with predefined templates

In [ ]:
# clear your state if running notebook more than once
from helper import reset_lesson
reset_lesson(4)

## What you'll build

A chat interface where the agent composes rich UI surfaces — dashboards, charts, and flight carousels — from a catalog of reusable components:

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Show me a sales dashboard with total revenue, new customers, and conversion rate metrics.</div>

<img src="images/a2ui-dynamic-dashboard.png" alt="A2UI Dynamic Dashboard" style="max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

## What is Declarative Generative UI?
Declarative Generative UI lets you define a set of UI building blocks that the agent composes into interfaces. It selects components from a catalog, arranges them into a schema, and binds runtime data to fill in the result.

Three pieces make this work:
* **Component catalog**: the UI primitives your app supports, split into two parts:
  * **Definitions**: platform-agnostic descriptions of each component's name, props, and purpose.
  * **Renderers**: platform-specific implementations that turn definitions into actual UI (e.g. React components).
* **Schema**: the structured description of which components to use, how they nest, and how they relate.
* **Data bindings**: the runtime values that populate the schema with real content like flight details, metrics, or records.

A useful mental model is Lego: the **catalog** is the box of pieces, the **schema** is how they snap together, and the **data bindings** fill in the final details at runtime. The agent assembles interfaces dynamically while your app keeps control over consistency, safety, and rendering quality.

<img src="images/a2ui-overview.png" alt="A2UI Overview: components, schema, data bindings, and rendered UI" style="max-width: 700px; border: 1px solid #ddd; border-radius: 8px;" />

## Why use Declarative Generative UI?
Controlled Generative UI (L3) works well for your highest-traffic surfaces where predictability matters most. But in the long tail (internal tools, edge cases, varied user goals), hand-authoring every layout doesn't scale. That's where Declarative Generative UI fits: the agent assembles UI from a fixed set of building blocks, so you get adaptability without sacrificing safety or consistency.

**Pros**
- **Constrained flexibility:** the agent adapts the interface without going outside your component system.
- **Bring your own components:** you define the primitives, the agent decides how to combine them.
- **Less work per surface:** define the catalog once, reuse it everywhere.
- **Cross-platform by design:** the same schema renders across web, mobile, Slack, and text messaging.
- **More token-efficient than open generative UI:** the agent works from a fixed vocabulary instead of generating arbitrary code.

**Cons**
- **Less pixel-perfect control:** you can't fine-tune the exact final interface.
- **Less predictable:** the agent may assemble components differently in similar situations.
- **More error-prone:** schemas and data bindings can fail in subtle ways, requiring validation and recovery logic.
- **Requires upfront design:** the component catalog, schema format, and renderer contracts need careful definition.

In [ ]:
# supress warning messages
import warnings
warnings.filterwarnings("ignore")

### Setup Dependencies

In [ ]:
# install dependencies
# !pip install -r requirements.txt

In [ ]:
from helper import install_frontend
install_frontend()

In [ ]:
import os
from helper import get_openai_api_key  # platform helper

os.environ["OPENAI_API_KEY"] = get_openai_api_key()
print("✓ OpenAI API key loaded")

## Build the agent

[A2UI](https://a2ui.org/) (Agent-to-UI) is a specification created by Google for declarative generative UI. CopilotKit and AG-UI support A2UI natively — CopilotKit maintains the A2UI React renderer used in this lesson.

### Start the server

Start a FastAPI server with a placeholder agent graph, same pattern as L2:

In [ ]:
from ag_ui_langgraph import add_langgraph_fastapi_endpoint
from copilotkit import LangGraphAGUIAgent
from langchain.agents import create_agent
from fastapi import FastAPI
from helper import start_server

app = FastAPI()

# Create agent with placeholder graph (set in the next cell)
graph = create_agent("openai:gpt-4.1")
agent = LangGraphAGUIAgent(
    name="lesson4_agent",
    description="Lesson 4 A2UI agent",
    graph=graph
)

add_langgraph_fastapi_endpoint(app=app, agent=agent, path="/")

start_server(app, port=8004)

### Define agent

Here you'll enable A2UI generation through `CopilotKitMiddleware`. When A2UI is enabled, CopilotKit automatically adds the backend behavior needed to generate structured A2UI output — you don't implement that flow manually.

Under the hood, this works through two layers of tool calls:
- An **outer tool call** fires when the agent decides to generate A2UI — this keeps A2UI-specific logic separate from the rest of the agent's behavior.
- An **inner tool call** contains the structured A2UI payload. The middleware intercepts these arguments to enable streaming.

The generated output is also included in the agent history as a tool call result.

You'll also add a `get_sales_data` tool — a traditional data-fetching tool that returns sales metrics. This shows that A2UI works alongside regular tools: the agent first fetches data, then visualizes it with `generate_a2ui`.

In [ ]:
import json
from typing import Any

from copilotkit import CopilotKitMiddleware
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool as lc_tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver


# ── Data-fetching tool (placeholder for a real database/API call) ────
@tool
def get_sales_data() -> str:
    """Fetch current sales metrics and revenue data.

    Returns sales data including revenue, customers, conversion rates,
    and breakdowns by category and month.
    """
    # Placeholder: in production, this would query your actual database or API.
    return json.dumps({
        "totalRevenue": "$1.2M",
        "newCustomers": 3842,
        "conversionRate": "3.6%",
        "revenueByCategory": [
            {"label": "Electronics", "value": 420000},
            {"label": "Clothing", "value": 310000},
            {"label": "Home & Garden", "value": 185000},
            {"label": "Sports", "value": 160000},
            {"label": "Books", "value": 125000},
        ],
        "monthlySales": [
            {"label": "Jan", "value": 85000},
            {"label": "Feb", "value": 92000},
            {"label": "Mar", "value": 108000},
            {"label": "Apr", "value": 95000},
            {"label": "May", "value": 115000},
            {"label": "Jun", "value": 125000},
        ],
    })

In [ ]:
graph = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[get_sales_data],
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt=(
        "You are a helpful assistant that creates rich visual UI.\n\n"
        "Tool guidance:\n"
        "- For sales/business data requests: first call get_sales_data to fetch "
        "the latest metrics, then call generate_a2ui to visualize the results "
        "as a dashboard with charts, metrics, and cards.\n"
        "- For other rich UI: call generate_a2ui directly.\n\n"
        "IMPORTANT: After calling a tool, do NOT repeat or summarize the data "
        "in your text response. The tool renders UI automatically. "
        "Just confirm what was rendered."
    ),
)

agent.graph = graph
print("✓ Agent graph updated!")

## Add A2UI rendering to the frontend

The frontend has two key pieces: the **CopilotKit runtime** endpoint and the **component catalog** that powers Declarative Generative UI.

### CopilotKit Runtime

Like in Lesson 2, the frontend needs a CopilotKit runtime endpoint that bridges the UI to the backend agent.

The key difference in this lesson is the `a2ui` configuration — `injectA2UITool: true` tells the runtime to automatically inject an A2UI tool that the agent can call to generate declarative UI.

In [ ]:
%%writefile frontend/server.ts

import { serve } from "@hono/node-server";
import {
  CopilotRuntime,
  createCopilotEndpoint,
} from "@copilotkit/runtime/v2";
import { LangGraphHttpAgent } from "@copilotkit/runtime/langgraph";

const langGraphAgent = new LangGraphHttpAgent({ url: "http://localhost:8004" });

const runtime = new CopilotRuntime({
  agents: { default: langGraphAgent },
  a2ui: { injectA2UITool: true },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4004 }, () => {
  console.log("\u2713 CopilotKit API server running at http://localhost:4004");
});

### Component Catalog — Definitions

A component definition is the contract for a primitive in your catalog:
- a **name** the agent can reference (like Card, Row, or Text)
- a **schema** for its props
- and the **React component** that draws it in your app's styles

Component definitions live in the renderer setup, where CopilotKit's A2UI integration uses them as a registry. 

In [ ]:
%%writefile frontend/src/catalog/definitions.ts

import { z } from "zod";

export const demonstrationCatalogDefinitions = {
  Title: {
    description: "A heading. Use for section titles and page headers.",
    props: z.object({
      text: z.string(),
      level: z.string().optional(),
    }),
  },

  Text: {
    description: "A text element. Use for labels, values, captions.",
    props: z.object({
      text: z.union([z.string(), z.object({ path: z.string() })]),
      variant: z.enum(["h1", "h2", "h3", "body", "caption"]).optional(),
    }),
  },

  Icon: {
    description: "A material icon by name.",
    props: z.object({
      name: z.string(),
      size: z.number().optional(),
    }),
  },

  Image: {
    description: "An image element.",
    props: z.object({
      src: z.union([z.string(), z.object({ path: z.string() })]),
      alt: z.union([z.string(), z.object({ path: z.string() })]).optional(),
      width: z.number().optional(),
      height: z.number().optional(),
    }),
  },

  Divider: {
    description: "A horizontal divider line.",
    props: z.object({}),
  },

  Card: {
    description: "A generic card container with a child slot.",
    props: z.object({
      child: z.string().optional(),
    }),
  },

  List: {
    description: "A list of children. Supports horizontal or vertical direction.",
    props: z.object({
      children: z.union([
        z.array(z.string()),
        z.object({ componentId: z.string(), path: z.string() }),
      ]),
      direction: z.enum(["horizontal", "vertical"]).optional(),
      gap: z.number().optional(),
    }),
  },

  Tabs: {
    description: "A tabbed container. Each tab has a label and child content.",
    props: z.object({
      tabs: z.array(z.object({ label: z.string(), child: z.string() })),
    }),
  },

  Row: {
    description: "Horizontal layout container.",
    props: z.object({
      gap: z.number().optional(),
      align: z.string().optional(),
      justify: z.string().optional(),
      children: z.union([
        z.array(z.string()),
        z.object({ componentId: z.string(), path: z.string() }),
      ]),
    }),
  },

  Column: {
    description: "Vertical layout container.",
    props: z.object({
      gap: z.number().optional(),
      align: z.string().optional(),
      children: z.union([
        z.array(z.string()),
        z.object({ componentId: z.string(), path: z.string() }),
      ]),
    }),
  },

  DashboardCard: {
    description:
      "A card container with title and optional subtitle. Has a 'child' slot for content (chart, metrics, etc). Use 'child' with a single component ID.",
    props: z.object({
      title: z.string(),
      subtitle: z.string().optional(),
      child: z.string().optional(),
    }),
  },

  Metric: {
    description:
      "A key metric display with label, value, and optional trend indicator. Great for KPIs and stats.",
    props: z.object({
      label: z.string(),
      value: z.string(),
      trend: z.enum(["up", "down", "neutral"]).optional(),
      trendValue: z.string().optional(),
    }),
  },

  PieChart: {
    description:
      "A pie/donut chart. Provide data as array of {label, value, color} objects.",
    props: z.object({
      data: z.array(
        z.object({
          label: z.string(),
          value: z.number(),
          color: z.string().optional(),
        }),
      ),
      innerRadius: z.number().optional(),
    }),
  },

  BarChart: {
    description:
      "A bar chart. Provide data as array of {label, value} objects.",
    props: z.object({
      data: z.array(z.object({ label: z.string(), value: z.number() })),
      color: z.string().optional(),
    }),
  },

  Badge: {
    description:
      "A small status badge/tag. Use for labels, statuses, categories.",
    props: z.object({
      text: z.string(),
      variant: z
        .enum(["success", "warning", "error", "info", "neutral"])
        .optional(),
    }),
  },

  DataTable: {
    description: "A data table with columns and rows.",
    props: z.object({
      columns: z.array(z.object({ key: z.string(), label: z.string() })),
      rows: z.array(z.record(z.any())),
    }),
  },

  Button: {
    description:
      "An interactive button. Use 'label' for simple text or 'child' for a child component. 'action' is dispatched on click.",
    props: z.object({
      label: z.string().optional(),
      child: z
        .string()
        .describe(
          "The ID of the child component (e.g. a Text component for the label).",
        )
        .optional(),
      variant: z.enum(["primary", "secondary", "ghost"]).optional(),
      action: z
        .union([
          z.object({
            event: z.object({
              name: z.string(),
              context: z.record(z.any()).optional(),
            }),
          }),
          z.null(),
        ])
        .optional(),
    }),
  },
};

/** Type helper for renderers */
export type DemonstrationCatalogDefinitions =
  typeof demonstrationCatalogDefinitions;

### Component Catalog — Renderers

Renderers are platform-specific implementations that turn definitions into actual UI. Each renderer maps a component name to a React component, type-checked against the Zod schemas from the definitions.

At the bottom, the catalog is assembled with `createCatalog()` and registered in `main.tsx` via:

```tsx
<CopilotKitProvider a2ui={{ catalog: demonstrationCatalog }}>
```

This is what connects the agent's A2UI output to your React rendering layer.

In [ ]:
%%writefile frontend/src/catalog/renderers.tsx

import React from "react";
import {
  PieChart as RechartsPie,
  Pie,
  Cell,
  ResponsiveContainer,
  BarChart as RechartsBar,
  Bar,
  XAxis,
  YAxis,
  Tooltip,
  CartesianGrid,
} from "recharts";
import {
  createCatalog,
  type CatalogRenderers,
} from "@copilotkit/a2ui-renderer";
import {
  demonstrationCatalogDefinitions,
  type DemonstrationCatalogDefinitions,
} from "./definitions";

function resolveText(value: unknown): string {
  if (typeof value === "string") return value;
  if (value && typeof value === "object" && "path" in value)
    return String((value as { path: string }).path);
  return String(value ?? "");
}

// ─── Renderers (type-checked against schema definitions) ────────────

const demonstrationCatalogRenderers: CatalogRenderers<DemonstrationCatalogDefinitions> =
  {
    Title: ({ props }) => {
      const Tag = (
        props.level === "h1" ? "h1" : props.level === "h3" ? "h3" : "h2"
      ) as "h1" | "h2" | "h3";
      const sizes: Record<string, string> = {
        h1: "1.75rem",
        h2: "1.25rem",
        h3: "1rem",
      };
      return (
        <Tag
          style={{
            margin: 0,
            fontWeight: 600,
            fontSize: sizes[props.level ?? "h2"],
            color: "#111827",
            letterSpacing: "-0.01em",
          }}
        >
          {resolveText(props.text)}
        </Tag>
      );
    },

    Text: ({ props }) => {
      const styles: Record<string, React.CSSProperties> = {
        h1: { fontSize: "1.5rem", fontWeight: 700, color: "#111827" },
        h2: { fontSize: "1.25rem", fontWeight: 700, color: "#111827" },
        h3: { fontSize: "1rem", fontWeight: 600, color: "#374151" },
        body: { fontSize: "0.875rem", color: "#374151" },
        caption: { fontSize: "0.75rem", color: "#6b7280" },
      };
      return (
        <span style={styles[props.variant ?? "body"]}>
          {resolveText(props.text)}
        </span>
      );
    },

    Icon: ({ props }) => (
      <span
        className="material-symbols-outlined"
        style={{ fontSize: props.size ?? 24, color: "#6b7280" }}
      >
        {props.name}
      </span>
    ),

    Image: ({ props }) => (
      <img
        src={resolveText(props.src)}
        alt={resolveText(props.alt ?? "")}
        style={{
          width: props.width ?? "auto",
          height: props.height ?? "auto",
          maxWidth: "100%",
          borderRadius: 8,
        }}
      />
    ),

    Divider: () => (
      <hr style={{ border: "none", borderTop: "1px solid #e5e7eb", margin: "4px 0" }} />
    ),

    Card: ({ props, children }) => (
      <div
        style={{
          background: "#fff",
          borderRadius: 12,
          border: "1px solid #e5e7eb",
          padding: 16,
          boxShadow: "0 1px 3px rgba(0,0,0,0.04)",
        }}
      >
        {typeof props.child === "string" && children(props.child)}
      </div>
    ),

    List: ({ props, children }) => {
      const items = Array.isArray(props.children) ? props.children : [];
      const isHorizontal = (props as any).direction === "horizontal";
      return (
        <div
          style={{
            display: "flex",
            flexDirection: isHorizontal ? "row" : "column",
            gap: props.gap ?? 8,
            overflowX: isHorizontal ? "auto" : undefined,
            flexWrap: isHorizontal ? "nowrap" : undefined,
          }}
        >
          {items.map((item: any, i: number) => {
            if (typeof item === "string")
              return <React.Fragment key={`${item}-${i}`}>{children(item)}</React.Fragment>;
            if (item && typeof item === "object" && "id" in item)
              return (
                <div key={`${item.id}-${i}`} style={isHorizontal ? { flex: "0 0 auto", minWidth: 280 } : undefined}>
                  {(children as any)(item.id, item.basePath)}
                </div>
              );
            return null;
          })}
        </div>
      );
    },

    Tabs: ({ props, children }) => {
      const [active, setActive] = React.useState(0);
      const tabs = props.tabs ?? [];
      return (
        <div>
          <div style={{ display: "flex", gap: 0, borderBottom: "1px solid #e5e7eb" }}>
            {tabs.map((tab: any, i: number) => (
              <button
                key={i}
                onClick={() => setActive(i)}
                style={{
                  padding: "8px 16px",
                  fontSize: "0.85rem",
                  fontWeight: active === i ? 600 : 400,
                  color: active === i ? "#111827" : "#6b7280",
                  borderBottom: active === i ? "2px solid #111827" : "2px solid transparent",
                  background: "none",
                  border: "none",
                  cursor: "pointer",
                }}
              >
                {tab.label}
              </button>
            ))}
          </div>
          <div style={{ padding: "12px 0" }}>
            {tabs[active] && children(tabs[active].child)}
          </div>
        </div>
      );
    },

    Row: ({ props, children }) => {
      const justifyMap: Record<string, string> = {
        start: "flex-start",
        center: "center",
        end: "flex-end",
        spaceBetween: "space-between",
      };
      const items = Array.isArray(props.children) ? props.children : [];
      return (
        <div
          style={{
            display: "flex",
            flexDirection: "row",
            gap: `${props.gap ?? 16}px`,
            alignItems: props.align ?? "stretch",
            justifyContent:
              justifyMap[props.justify ?? "start"] ?? "flex-start",
            flexWrap: "wrap",
            width: "100%",
          }}
        >
          {items.map((item: any, i: number) => {
            if (typeof item === "string")
              return (
                <div
                  key={`${item}-${i}`}
                  style={{ flex: "1 1 0", minWidth: 0 }}
                >
                  {children(item)}
                </div>
              );
            if (item && typeof item === "object" && "id" in item)
              return (
                <div
                  key={`${item.id}-${i}`}
                  style={{ flex: "1 1 0", minWidth: 0 }}
                >
                  {(children as any)(item.id, item.basePath)}
                </div>
              );
            return null;
          })}
        </div>
      );
    },

    Column: ({ props, children }) => {
      const items = Array.isArray(props.children) ? props.children : [];
      return (
        <div
          style={{
            display: "flex",
            flexDirection: "column",
            gap: `${props.gap ?? 12}px`,
            width: "100%",
          }}
        >
          {items.map((item: any, i: number) => {
            if (typeof item === "string")
              return (
                <React.Fragment key={`${item}-${i}`}>
                  {children(item)}
                </React.Fragment>
              );
            if (item && typeof item === "object" && "id" in item)
              return (
                <React.Fragment key={`${item.id}-${i}`}>
                  {(children as any)(item.id, item.basePath)}
                </React.Fragment>
              );
            return null;
          })}
        </div>
      );
    },

    DashboardCard: ({ props, children }) => (
      <div
        style={{
          background: "#fff",
          borderRadius: "12px",
          border: "1px solid #e5e7eb",
          padding: "20px",
          boxShadow: "0 1px 3px rgba(0,0,0,0.04)",
          display: "flex",
          flexDirection: "column",
          gap: "12px",
        }}
      >
        <div>
          <div
            style={{ fontWeight: 600, fontSize: "0.9rem", color: "#111827" }}
          >
            {resolveText(props.title)}
          </div>
          {props.subtitle && (
            <div
              style={{
                fontSize: "0.75rem",
                color: "#6b7280",
                marginTop: "2px",
              }}
            >
              {resolveText(props.subtitle)}
            </div>
          )}
        </div>
        {typeof props.child === "string" && children(props.child)}
      </div>
    ),

    Metric: ({ props }) => {
      const trendColors: Record<string, string> = {
        up: "#059669",
        down: "#dc2626",
        neutral: "#6b7280",
      };
      const trendIcons: Record<string, string> = {
        up: "↑",
        down: "↓",
        neutral: "→",
      };
      return (
        <div style={{ display: "flex", flexDirection: "column", gap: "4px" }}>
          <span
            style={{
              fontSize: "0.75rem",
              color: "#6b7280",
              fontWeight: 500,
              textTransform: "uppercase",
              letterSpacing: "0.05em",
            }}
          >
            {resolveText(props.label)}
          </span>
          <div style={{ display: "flex", alignItems: "baseline", gap: "8px" }}>
            <span
              style={{
                fontSize: "1.5rem",
                fontWeight: 700,
                color: "#111827",
                letterSpacing: "-0.02em",
              }}
            >
              {resolveText(props.value)}
            </span>
            {props.trend && props.trendValue && (
              <span
                style={{
                  fontSize: "0.8rem",
                  fontWeight: 500,
                  color: trendColors[props.trend] ?? "#6b7280",
                }}
              >
                {trendIcons[props.trend]} {resolveText(props.trendValue)}
              </span>
            )}
          </div>
        </div>
      );
    },

    PieChart: ({ props }) => {
      const COLORS = [
        "#3b82f6",
        "#8b5cf6",
        "#ec4899",
        "#f59e0b",
        "#10b981",
        "#6366f1",
      ];
      const data = props.data ?? [];
      return (
        <div style={{ width: "100%", height: 200 }}>
          <ResponsiveContainer>
            <RechartsPie>
              <Pie
                data={data}
                dataKey="value"
                nameKey="label"
                cx="50%"
                cy="50%"
                innerRadius={props.innerRadius ?? 40}
                outerRadius={80}
                paddingAngle={2}
              >
                {data.map((entry: any, i: number) => (
                  <Cell
                    key={i}
                    fill={entry.color ?? COLORS[i % COLORS.length]}
                  />
                ))}
              </Pie>
              <Tooltip />
            </RechartsPie>
          </ResponsiveContainer>
        </div>
      );
    },

    BarChart: ({ props }) => {
      const data = props.data ?? [];
      return (
        <div style={{ width: "100%", height: 200 }}>
          <ResponsiveContainer>
            <RechartsBar data={data}>
              <CartesianGrid strokeDasharray="3 3" stroke="#f3f4f6" />
              <XAxis dataKey="label" tick={{ fontSize: 11, fill: "#6b7280" }} />
              <YAxis tick={{ fontSize: 11, fill: "#6b7280" }} />
              <Tooltip />
              <Bar
                dataKey="value"
                fill={props.color ?? "#3b82f6"}
                radius={[4, 4, 0, 0]}
              />
            </RechartsBar>
          </ResponsiveContainer>
        </div>
      );
    },

    Badge: ({ props }) => {
      const variants: Record<string, { bg: string; color: string }> = {
        success: { bg: "#dcfce7", color: "#166534" },
        warning: { bg: "#fef3c7", color: "#92400e" },
        error: { bg: "#fee2e2", color: "#991b1b" },
        info: { bg: "#dbeafe", color: "#1e40af" },
        neutral: { bg: "#f3f4f6", color: "#374151" },
      };
      const v = variants[props.variant ?? "neutral"] ?? variants.neutral;
      return (
        <span
          style={{
            display: "inline-block",
            padding: "2px 8px",
            borderRadius: "9999px",
            fontSize: "0.7rem",
            fontWeight: 500,
            background: v.bg,
            color: v.color,
          }}
        >
          {resolveText(props.text)}
        </span>
      );
    },

    DataTable: ({ props }) => {
      const cols = props.columns ?? [];
      const rows = props.rows ?? [];
      return (
        <div style={{ overflowX: "auto", width: "100%" }}>
          <table
            style={{
              width: "100%",
              borderCollapse: "collapse",
              fontSize: "0.8rem",
            }}
          >
            <thead>
              <tr>
                {cols.map((col: any) => (
                  <th
                    key={col.key}
                    style={{
                      textAlign: "left",
                      padding: "8px 12px",
                      borderBottom: "2px solid #e5e7eb",
                      color: "#6b7280",
                      fontWeight: 600,
                      fontSize: "0.7rem",
                      textTransform: "uppercase",
                      letterSpacing: "0.05em",
                    }}
                  >
                    {col.label}
                  </th>
                ))}
              </tr>
            </thead>
            <tbody>
              {rows.map((row: any, i: number) => (
                <tr key={i} style={{ borderBottom: "1px solid #f3f4f6" }}>
                  {cols.map((col: any) => (
                    <td
                      key={col.key}
                      style={{ padding: "8px 12px", color: "#374151" }}
                    >
                      {String(row[col.key] ?? "")}
                    </td>
                  ))}
                </tr>
              ))}
            </tbody>
          </table>
        </div>
      );
    },

    Button: ({ props, children, dispatch }) => {
      const variants: Record<string, React.CSSProperties> = {
        primary: { background: "#111827", color: "#fff", border: "none" },
        secondary: {
          background: "#fff",
          color: "#374151",
          border: "1px solid #d1d5db",
        },
        ghost: { background: "transparent", color: "#3b82f6", border: "none" },
      };
      const style = variants[props.variant ?? "primary"] ?? variants.primary;
      return (
        <button
          style={{
            ...style,
            padding: "8px 16px",
            borderRadius: "8px",
            fontSize: "0.8rem",
            fontWeight: 500,
            cursor: "pointer",
            transition: "opacity 0.15s",
            width: "100%",
          }}
          onClick={() => dispatch?.(props.action)}
        >
          {typeof props.child === "string" ? children(props.child) : (props as any).label ?? null}
        </button>
      );
    },
  };

// ─── Assembled Catalog ───────────────────────────────────────────────

export const demonstrationCatalog = createCatalog(
  demonstrationCatalogDefinitions,
  demonstrationCatalogRenderers,
  {
    catalogId: "copilotkit://app-dashboard-catalog",
    includeBasicCatalog: false,
  },
);

### Wiring it together — `main.tsx` and `App.tsx`

In `main.tsx`, the `CopilotKitProvider` connects to the runtime and registers the catalog via `a2ui={{ catalog: demonstrationCatalog }}`.

In `App.tsx`, a `CopilotChat` component points at the `default` agent — the same pattern from Lesson 2.

In [ ]:
%%writefile frontend/src/main.tsx

import { StrictMode } from "react";
import { createRoot } from "react-dom/client";
import { CopilotKit } from "@copilotkit/react-core/v2";
import "@copilotkit/react-core/v2/styles.css";
import { demonstrationCatalog } from "./catalog/renderers";
import "./globals.css";
import App from "./App";

createRoot(document.getElementById("root")!).render(
  <StrictMode>
    <main className="h-screen w-screen">
      <CopilotKit
        useSingleEndpoint={false}
        runtimeUrl="/api/copilotkit"

        // 🪁 Enable A2UI using the demonstration catalog
        a2ui={{ catalog: demonstrationCatalog }}
      >
        <App />
      </CopilotKit>
    </main>
  </StrictMode>,
);

## Add a suggestion button for sales dashboard
Add a suggestion for the Dynamic Schema component "Sales Dashboard" button. 

In [ ]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";
import { useExampleDynamicSuggestions } from "@/hooks/use-example-suggestions";

export default function App() {
  useExampleDynamicSuggestions();
  return <CopilotChat />;
}

## Start the frontend

Run the next two cells to start the dev server and open a live preview.

In [ ]:
from helper import start_frontend
start_frontend(port=3004)

### Display the app

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Sales Dashboard</div>

In [ ]:
from helper import display_app
display_app(port=3004)

## Fixed Schema Declarative Generative UI

So far, you've built a **dynamic schema** implementation where the agent generates the A2UI component tree on the fly. But there's an alternative approach: **fixed schema** declarative generative UI.

With a fixed schema, you design the A2UI component tree ahead of time — the layout, nesting, and data bindings are all predefined. The agent's only job is to fill in the runtime data. This gives you maximum control over the final UI while still letting the agent drive when and what data to show.

This is useful when you want a consistent, polished layout for a specific surface (e.g. a flight card carousel) and don't need the agent to improvise the structure.

### A2UI Composer

The easiest way to build a fixed schema is to use the [A2UI Composer](https://a2ui-editor.ag-ui.com/). Open the Composer and specify the following prompt:

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Create a carousel of flight cards with origin, destination, duration, time of departure, and time of arrival</div>

![A2UI Composer](images/a2ui-composer.png)

Then click the `Copy JSON` button to copy the schema. 

You'll use this schema when you define the `generate_schema` tool.

![A2UI Composer Schema](images/a2ui-composer-schema.png)

The composer generates this schema for you. In production you would iterate on the schema in the Composer, preview it, and paste the final JSON in your tool. 

## Adding a fixed schema

In AG-UI, **any tool** can return A2UI operations — just wrap the result in an `a2ui_operations` array. The runtime's A2UI middleware detects these operations and routes them to the frontend for rendering.

Here, you define a `display_flights` tool that returns a **predefined** A2UI component tree with runtime flight data merged in. The schema (produced by the A2UI Composer) is fixed — only the data changes per invocation.

You then add this tool alongside the existing `generate_a2ui` tool. The agent decides which to call: `generate_a2ui` for open-ended dashboards, `display_flights` for the fixed flight card layout.

<img src="images/a2ui-fixed-carousel.png" alt="A2UI Fixed Schema Flight Carousel" style="max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

### Adding a fixed schema tool

In [ ]:
import logging; logging.getLogger("langgraph.checkpoint.serde.jsonplus").setLevel(logging.ERROR)
import json
from typing_extensions import TypedDict

from copilotkit import a2ui
from langchain.tools import tool

CATALOG_ID = "copilotkit://app-dashboard-catalog"
SURFACE_ID = "flight-search-results"

FLIGHT_SCHEMA = [
    {"id": "root", "component": "List", "children": {"componentId": "flight-card", "path": "/flights"}, "direction": "horizontal", "gap": 16},
    {"id": "flight-card", "component": "Card", "child": "main-col"},
    {"id": "main-col", "component": "Column", "children": ["airline-img", "header-row", "meta-row", "divider-1", "times-row", "route-row", "divider-2", "status-row", "divider-3", "book-btn"], "align": "stretch", "gap": 8},
    {"id": "airline-img", "component": "Image", "src": {"path": "airlineLogo"}, "alt": {"path": "airline"}, "height": 32},
    {"id": "header-row", "component": "Row", "children": ["airline-name", "price-text"], "justify": "spaceBetween", "align": "center"},
    {"id": "airline-name", "component": "Text", "text": {"path": "airline"}, "variant": "h3"},
    {"id": "price-text", "component": "Text", "text": {"path": "price"}, "variant": "h2"},
    {"id": "meta-row", "component": "Row", "children": ["flight-number", "date-text"], "justify": "spaceBetween", "align": "center"},
    {"id": "flight-number", "component": "Text", "text": {"path": "flightNumber"}, "variant": "caption"},
    {"id": "date-text", "component": "Text", "text": {"path": "date"}, "variant": "caption"},
    {"id": "divider-1", "component": "Divider"},
    {"id": "times-row", "component": "Row", "children": ["depart-time", "duration-text", "arrive-time"], "justify": "spaceBetween", "align": "center"},
    {"id": "depart-time", "component": "Text", "text": {"path": "departureTime"}, "variant": "h2"},
    {"id": "duration-text", "component": "Text", "text": {"path": "duration"}, "variant": "caption"},
    {"id": "arrive-time", "component": "Text", "text": {"path": "arrivalTime"}, "variant": "h2"},
    {"id": "route-row", "component": "Row", "children": ["origin-code", "arrow-text", "dest-code"], "justify": "spaceBetween", "align": "center"},
    {"id": "origin-code", "component": "Text", "text": {"path": "origin"}, "variant": "h3"},
    {"id": "arrow-text", "component": "Text", "text": "\u2192", "variant": "h3"},
    {"id": "dest-code", "component": "Text", "text": {"path": "destination"}, "variant": "h3"},
    {"id": "divider-2", "component": "Divider"},
    {"id": "status-row", "component": "Row", "children": ["status-text"], "align": "center"},
    {"id": "status-text", "component": "Text", "text": {"path": "status"}, "variant": "caption"},
    {"id": "divider-3", "component": "Divider"},
    {"id": "book-btn", "component": "Button", "label": "Book Flight", "variant": "primary", "action": {"event": {"name": "bookFlight"}}},
]

In [ ]:
class Flight(TypedDict):
    id: str
    airline: str
    airlineLogo: str
    flightNumber: str
    origin: str
    destination: str
    date: str
    departureTime: str
    arrivalTime: str
    duration: str
    status: str
    price: str
    
# ── Data-fetching tool (placeholder for a real flight search API) ────
@tool
def search_flights(origin: str, destination: str) -> list[Flight]:
    """Search for available flights between two airports.

    Args:
        origin: Origin airport IATA code (e.g. "SFO").
        destination: Destination airport IATA code (e.g. "JFK").
    """
    # Placeholder: in production, this would call a real flight search API.
    return [
        {"id": "1", "airline": "Delta Air Lines", "airlineLogo": f"https://www.gstatic.com/flights/airline_logos/70px/DL.png", "flightNumber": "DL 520", "origin": origin, "destination": destination, "date": "2026-04-11", "departureTime": "08:00", "arrivalTime": "16:35", "duration": "5h 35m", "status": "On Time", "price": "$389"},
        {"id": "2", "airline": "United Airlines", "airlineLogo": f"https://www.gstatic.com/flights/airline_logos/70px/UA.png", "flightNumber": "UA 1583", "origin": origin, "destination": destination, "date": "2026-04-11", "departureTime": "10:15", "arrivalTime": "18:42", "duration": "5h 27m", "status": "On Time", "price": "$412"},
        {"id": "3", "airline": "JetBlue", "airlineLogo": f"https://www.gstatic.com/flights/airline_logos/70px/B6.png", "flightNumber": "B6 416", "origin": origin, "destination": destination, "date": "2026-04-11", "departureTime": "14:30", "arrivalTime": "23:05", "duration": "5h 35m", "status": "On Time", "price": "$345"},
        {"id": "4", "airline": "American Airlines", "airlineLogo": f"https://www.gstatic.com/flights/airline_logos/70px/AA.png", "flightNumber": "AA 178", "origin": origin, "destination": destination, "date": "2026-04-11", "departureTime": "17:00", "arrivalTime": "01:20+1", "duration": "5h 20m", "status": "On Time", "price": "$398"},
    ]

In [ ]:
@tool
def display_flights(flights: list[Flight]) -> str:
    """Display flights as rich cards in a horizontal row.

    Each flight must have: id, airline, airlineLogo (URL), flightNumber,
    origin, destination, date, departureTime, arrivalTime, duration,
    status, and price.
    """
    return a2ui.render(
        operations=[
            a2ui.create_surface(SURFACE_ID, catalog_id=CATALOG_ID),
            a2ui.update_components(SURFACE_ID, FLIGHT_SCHEMA),
            a2ui.update_data_model(SURFACE_ID, {"flights": flights}),
        ],
    )

In [ ]:
# Re-create graph with both tools: dynamic (generate_a2ui) + fixed (search_flights)
graph = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[get_sales_data, search_flights, display_flights],
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt=(
        "You are a helpful assistant that creates rich visual UI.\n\n"
        "Tool guidance:\n"
        "- ALL flight-related queries: first call search_flights to fetch flight "
        "data, then call display_flights with the results. NEVER use generate_a2ui "
        "for flights.\n"
        "- For sales/business data requests: first call get_sales_data to fetch "
        "the latest metrics, then call generate_a2ui to visualize the results.\n"
        "- For other rich UI: call generate_a2ui directly.\n\n"
        "Airline logos: use https://www.gstatic.com/flights/airline_logos/70px/<IATA>.png\n"
        "Common codes: DL=Delta, UA=United, AA=American, WN=Southwest, B6=JetBlue, "
        "NK=Spirit, AS=Alaska, F9=Frontier, BA=British Airways, LH=Lufthansa, "
        "AF=Air France, EK=Emirates, QF=Qantas, SQ=Singapore Airlines, NH=ANA.\n\n"
        "IMPORTANT: After calling a tool, do NOT repeat or summarize the data "
        "in your text response. The tool renders UI automatically. "
        "Just confirm what was rendered."
    ),
)

agent.graph = graph
print("✓ Agent graph updated with display_flights tool!")

## Add a chat suggestion button for flights

Earlier, you added a suggestion for the Dynamic Schema components. Now, add a suggestion for the Fixed Schema component we just added. 

In [ ]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";
import { 
  useExampleDynamicSuggestions,
  useExampleFixedSuggestions
} from "@/hooks/use-example-suggestions";

export default function App() {
  useExampleDynamicSuggestions();
  useExampleFixedSuggestions();

  return <CopilotChat />;
}

## Display the app
<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Display flight information</div>

In [ ]:
from helper import display_app
display_app(port=3004)

### Dynamic vs Fixed: When to use which

| | **Fixed Schema** | **Dynamic Schema** |
|---|---|---|
| **Layout** | Predefined, identical every time | Agent-generated, varies per request |
| **Agent's role** | Fills in data only | Chooses components and layout |
| **Consistency** | Maximum | Varies |
| **Flexibility** | Minimal — new layouts require code changes | High — agent adapts to the request |
| **Best for** | Polished, known surfaces (flight cards, invoices) | Long-tail, exploratory, or internal surfaces |

In practice, many applications use both: fixed schemas for high-traffic, brand-sensitive surfaces, and dynamic schemas for everything else.

## What you learned

- Declarative Generative UI lets the agent compose interfaces from a catalog of building blocks — more flexible than controlled UI, more consistent than open-ended.
- The A2UI spec defines three pieces: a **component catalog** (definitions + renderers), a **schema** (how components are arranged), and **data bindings** (runtime values).
- **Dynamic schemas** let the agent generate the layout on the fly — good for long-tail and exploratory surfaces.
- **Fixed schemas** give you a predefined layout the agent populates with data — good for polished, high-traffic surfaces.
- Both approaches can coexist in the same agent.

## Next step

In **Lesson 5**, you'll move to **open-ended generative UI** — the far end of the spectrum. You'll connect an MCP App (Excalidraw) so the agent can launch full applications in the chat, then enable `openGenerativeUI` to let the agent generate arbitrary UI on the fly. 